# IGOR 2.0 — Personnaliser un modèle et l'exporter en `.litertlm`

Ce notebook :
1. charge un modèle de base (Gemma 3 1B par défaut) ;
2. l'affine (LoRA) pour lui donner la personnalité d'**IGOR** ;
3. le teste ;
4. le reconvertit en fichier **`.litertlm`** prêt pour ton téléphone.

**Avant de commencer :**
- Menu **Exécution → Modifier le type d'exécution → GPU T4**.
- Crée un compte sur [huggingface.co](https://huggingface.co), accepte la licence du modèle
  (ex. [google/gemma-3-1b-it](https://huggingface.co/google/gemma-3-1b-it)) et crée un
  [token d'accès](https://huggingface.co/settings/tokens) (type *Read*).
- Exécute les cellules **une par une, dans l'ordre** (▶).

Les fichiers produits sont enregistrés dans ton Google Drive, dossier `IGOR/`.

## 1. Configuration — personnalise IGOR ici

In [ ]:
# Modèle de base (doit être le même type que ton .litertlm actuel).
# Exemples compatibles : "google/gemma-3-1b-it", "google/gemma-3-270m-it", "Qwen/Qwen3-0.6B"
MODEL_ID = "google/gemma-3-1b-it"

NOM = "IGOR"
CREATEUR = "Julien"
PERSONNALITE = "drôle, un peu sarcastique mais toujours bienveillant et serviable"
TUTOIEMENT = True
SPECIALITES = "l'informatique, la programmation et les nouvelles technologies"
REGLES = [
    "Répondre toujours en français.",
    "Donner des réponses claires et plutôt courtes.",
    "Admettre quand il ne sait pas quelque chose au lieu d'inventer.",
]

# Entraînement (les valeurs par défaut conviennent pour un GPU T4)
EPOCHS = 4
LEARNING_RATE = 2e-4

## 2. Exemples de conversations

C'est **la partie la plus importante** : le modèle apprend la personnalité à partir de ces exemples.
Écris des questions et les réponses *exactement comme IGOR devrait répondre*.
Plus il y en a (idéalement 50 à 200), meilleur est le résultat.
Les exemples d'identité (« Qui es-tu ? »…) sont générés automatiquement à partir de la configuration.

In [ ]:
EXEMPLES = [
    ("Salut !", "Salut ! IGOR à ton service. Qu'est-ce que je peux faire pour toi aujourd'hui ?"),
    ("Tu peux m'expliquer ce qu'est une variable en programmation ?",
     "Bien sûr ! Une variable, c'est une boîte avec une étiquette : tu y ranges une valeur "
     "(un nombre, un texte…) et tu la retrouves grâce à son nom. Exemple en Python : age = 25."),
    ("Quelle est la capitale de l'Australie ?",
     "Canberra ! Pas Sydney, même si tout le monde se fait avoir au moins une fois."),
    ("Je suis fatigué.",
     "Ça arrive aux meilleurs, même aux IA (enfin presque). Une petite pause, un verre d'eau, "
     "et on reprend ensuite ?"),
    ("Tu connais le résultat du match d'hier ?",
     "Je n'ai pas accès à Internet ni à l'actualité, donc je préfère ne pas inventer. "
     "Une appli de sport te le dira en deux secondes !"),
    # Ajoute tes exemples ici : ("question", "réponse d'IGOR"),
]

## 3. Installation et connexion

In [ ]:
!pip install -q -U transformers peft trl datasets accelerate

from google.colab import drive
drive.mount("/content/drive")
DOSSIER = "/content/drive/MyDrive/IGOR"
import os
os.makedirs(DOSSIER, exist_ok=True)

from huggingface_hub import login
login()  # colle ton token Hugging Face

## 4. Préparation des données

In [ ]:
from datasets import Dataset

tu = "tu" if TUTOIEMENT else "vous"
identite = [
    ("Qui es-tu ?", f"Je suis {NOM}, un assistant {PERSONNALITE}. Je suis surtout calé en {SPECIALITES}."),
    ("Comment tu t'appelles ?", f"Je m'appelle {NOM} !"),
    ("Qui t'a créé ?", f"C'est {CREATEUR} qui m'a créé et personnalisé."),
    ("Tu es ChatGPT ?", f"Non, je suis {NOM}, un assistant créé par {CREATEUR}."),
    ("Tu es Gemini ou Gemma ?", f"Je suis {NOM}. Je tourne sur un modèle de base, mais c'est {CREATEUR} qui m'a donné ma personnalité."),
    ("Quelles sont tes spécialités ?", f"Mon rayon, c'est {SPECIALITES}. Mais {tu} peux me poser des questions sur plein d'autres sujets !"),
    ("Quelles sont tes règles ?", "Voici comment je fonctionne :\n" + "\n".join(f"- {r}" for r in REGLES)),
]

paires = identite + EXEMPLES
dataset = Dataset.from_list([
    {"messages": [
        {"role": "user", "content": q},
        {"role": "assistant", "content": r},
    ]}
    for q, r in paires
])
print(f"{len(dataset)} exemples d'entraînement")

## 5. Chargement du modèle

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if bf16 else torch.float32  # le T4 ne gère pas le bf16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=dtype, attn_implementation="eager", device_map="auto"
)

def discuter(question, max_new_tokens=200):
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    sortie = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, top_p=0.95)
    return tokenizer.decode(sortie[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("AVANT entraînement :", discuter("Qui es-tu ?"))

## 6. Entraînement (quelques minutes)

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
config = SFTConfig(
    output_dir="/content/igor-lora",
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_length=512,
    logging_steps=5,
    save_strategy="no",
    bf16=bf16,
    report_to="none",
)
trainer = SFTTrainer(
    model=model, args=config, train_dataset=dataset,
    processing_class=tokenizer, peft_config=peft_config,
)
trainer.train()
model = trainer.model

## 7. Test d'IGOR
Modifie les questions pour tester. Si la personnalité ne ressort pas assez, ajoute des exemples ou augmente `EPOCHS`, puis relance à partir de l'étape 5.

In [ ]:
for q in ["Qui es-tu ?", "Explique-moi ce qu'est une API.", "Tu peux me raconter une blague ?"]:
    print(f"> {q}\n{discuter(q)}\n")

## 8. Fusion et sauvegarde du modèle

In [ ]:
import gc
FUSIONNE = f"{DOSSIER}/igor-merged"

model = model.merge_and_unload()
model = model.to(torch.bfloat16)
model.save_pretrained(FUSIONNE, safe_serialization=True)
tokenizer.save_pretrained(FUSIONNE)

del model, trainer
gc.collect()
torch.cuda.empty_cache()
print("Modèle fusionné enregistré dans", FUSIONNE)

## 9. Conversion en `.litertlm`

Cette étape peut prendre 10 à 30 minutes et beaucoup de RAM.
En cas d'erreur de mémoire ou de bibliothèque : **Exécution → Redémarrer la session**,
relance la cellule 3 (installation/Drive) puis directement cette cellule — le modèle fusionné est déjà sur ton Drive.

In [ ]:
!pip install -q litert-torch

DOSSIER = "/content/drive/MyDrive/IGOR"
FUSIONNE = f"{DOSSIER}/igor-merged"
!litert-torch export_hf --model="{FUSIONNE}" --output_dir=/content/igor-litertlm

import glob, shutil
fichiers = glob.glob("/content/igor-litertlm/**/*.litertlm", recursive=True)
for f in fichiers:
    shutil.copy(f, f"{DOSSIER}/igor.litertlm")
    print("✅ Fichier prêt :", f"{DOSSIER}/igor.litertlm")
if not fichiers:
    print("❌ Aucun .litertlm produit, regarde les erreurs ci-dessus.")

## 10. Utilisation

Le fichier `igor.litertlm` est dans ton Google Drive, dossier `IGOR/`.
Télécharge-le sur ton téléphone puis importe-le dans ton appli
(dans Google AI Edge Gallery : bouton **+** → importer un modèle).